# Prompt generation: Program synthesis

A walk-through for CausalARC prompt generation.

Jacqueline Maasch | August 2025

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns
import json
import platform
import ast
from itertools import permutations,product
from ast import literal_eval
import os
from os import listdir
from os.path import isfile, join

# Custom modules.
os.chdir("../causal_arc")
from carc import CausalARC
from carc_utils import UtilsARC
from carc_augment import AugmentARC
from carc_tasks_logical import TaskLogical
from carc_tasks_extension import TaskExtension
from carc_tasks_order import TaskOrder
from carc_tasks_counting import TaskCounting
from carc_tasks_sprites import TaskSprites

# View versioning.
print("python version     :", platform.python_version())
print("numpy version      :", np.__version__)
print("pandas version     :", pd.__version__)
print("matplotlib version :", matplotlib.__version__)
print("seaborn version    :", sns.__version__)

python version     : 3.12.2
numpy version      : 1.26.4
pandas version     : 2.2.3
matplotlib version : 3.10.0
seaborn version    : 0.13.2


In [2]:
c = CausalARC()
u = UtilsARC()
a = AugmentARC()
tl = TaskLogical()
te = TaskExtension()
to = TaskOrder()
tc = TaskCounting()
ts = TaskSprites()
all_tasks_dict = dict()
all_samples_dict = dict()

# Define functions

In [3]:
def get_prompt_replicates(sample_dict: dict,
                          n_prompts: int = 6,
                          n_examples_in_context: int = 4) -> dict:

    # Get prompt replicates.
    l1_dict = dict()
    l3_dict = dict()
    for i in range(n_prompts):
        
        # Get L1 prompt.
        l1_prompt_dict = c.get_prompt(sample_dict, 
                                      n_examples = n_examples_in_context,
                                      counterfactuals = False,
                                      scm = False,
                                      n_counterfactuals = 0, 
                                      problem_type = "induction")
        l1_dict[f"Replicate {i}"] = l1_prompt_dict

        # Get L3 prompt.
        l3_prompt_dict = c.get_prompt(sample_dict, 
                                      n_examples = n_examples_in_context//2,
                                      counterfactuals = True,
                                      scm = False,
                                      n_counterfactuals = 1, 
                                      problem_type = "induction")
        l3_dict[f"Replicate {i}"] = l3_prompt_dict

    return {"L1": l1_dict, "L3": l3_dict}

# Get prompts

In [4]:
n_examples_in_context = 8
n_examples = n_examples_in_context*2
n_prompts = 5
scms = ["SCMm5ob", "SCMev5t", "SCMfwpq", "SCMz750"]
methods = [tc.task_SCMm5ob, tc.task_SCMev5t, te.task_SCMfwpq, te.task_SCMz750]

task_names = []
for scm,method in zip(scms,methods):

    current_task_dict = dict()

    task_name = scm
    task_names.append(task_name)
    print(f"\n-*- {task_name} -*-")

    # Get sample dictionary.
    sample_dict = method(n_examples = n_examples, # Total input-output pairs per sample.
                         plot = False,
                         plot_type = "input_output", # "single"
                         figsize = (5,2),
                         grid = True)
    all_samples_dict[task_name] = sample_dict

    # Get prompt replicates.
    prompt_replicates = get_prompt_replicates(sample_dict,
                                              n_prompts = n_prompts,
                                              n_examples_in_context = n_examples_in_context)
    all_tasks_dict[task_name] = prompt_replicates
    current_task_dict[task_name] = prompt_replicates

    print("\n\nL1 prompt")
    print(prompt_replicates["L1"]["Replicate 0"])

    print("\n\nL3 prompt")
    print(prompt_replicates["L3"]["Replicate 0"])

    #print("\n\nsample_dict")
    #display(sample_dict)

    # Export.
    with open(f"../data/program_synthesis/{scm}_nexamples{n_examples_in_context}.json", "w") as f:
        json.dump(current_task_dict, f, indent = 4) # indent for readability.

print("\nTotal tasks:", len(task_names))


-*- SCMm5ob -*-


L1 prompt
You must solve the following puzzle by discovering the deterministic rule that maps inputs to outputs. Both the inputs and outputs are 2D Python arrays of colored pixels. We provide example input-output pairs as demonstration. To solve the problem, express the deterministic rule as a Python program. Do not explain your reasoning, and only output a single Python program.
Example input-output arrays:
[[0, 0, 0, 0, 0, 9, 0, 4, 0, 3], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 0, 2, 0, 0, 0], [0, 0, 0, 0, 0, 9, 0, 0, 2, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3], [0, 9, 0, 0, 0, 4, 0, 0, 0, 2], [0, 0, 0, 0, 0, 9, 0, 2, 4, 0], [3, 0, 3, 0, 3, 0, 0, 0, 3, 0], [0, 0, 4, 3, 0, 0, 3, 0, 4, 0], [9, 3, 0, 0, 0, 0, 2, 0, 0, 0]] -> [[0, 0, 0, 3], [0, 0, 0, 3], [4, 0, 0, 3], [4, 0, 0, 3], [4, 2, 9, 3], [4, 2, 9, 3], [4, 2, 9, 3], [4, 2, 9, 3], [4, 2, 9, 3]]
Example input-output arrays:
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [9, 0, 0, 2, 0, 0, 9, 0, 0, 0], [0, 0, 0, 0, 0, 0, 2, 0,

# Export full JSON

In [5]:
with open(f"../data/program_synthesis/program_synthesis_nexamples{n_examples_in_context}.json", "w") as f:
    json.dump(all_tasks_dict, f, indent = 4) # indent for readability.

In [6]:
with open(f"../data/program_synthesis/program_synthesis_nexamples{n_examples_in_context}_raw_samples.json", "w") as f:
    json.dump(all_samples_dict, f, indent = 4) # indent for readability.

In [7]:
print("Total tasks:", len(all_tasks_dict.keys()))

Total tasks: 4


In [8]:
all_tasks_dict

{'SCMm5ob': {'L1': {'Replicate 0': 'You must solve the following puzzle by discovering the deterministic rule that maps inputs to outputs. Both the inputs and outputs are 2D Python arrays of colored pixels. We provide example input-output pairs as demonstration. To solve the problem, express the deterministic rule as a Python program. Do not explain your reasoning, and only output a single Python program.\nExample input-output arrays:\n[[0, 0, 0, 0, 0, 9, 0, 4, 0, 3], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 0, 2, 0, 0, 0], [0, 0, 0, 0, 0, 9, 0, 0, 2, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3], [0, 9, 0, 0, 0, 4, 0, 0, 0, 2], [0, 0, 0, 0, 0, 9, 0, 2, 4, 0], [3, 0, 3, 0, 3, 0, 0, 0, 3, 0], [0, 0, 4, 3, 0, 0, 3, 0, 4, 0], [9, 3, 0, 0, 0, 0, 2, 0, 0, 0]] -> [[0, 0, 0, 3], [0, 0, 0, 3], [4, 0, 0, 3], [4, 0, 0, 3], [4, 2, 9, 3], [4, 2, 9, 3], [4, 2, 9, 3], [4, 2, 9, 3], [4, 2, 9, 3]]\nExample input-output arrays:\n[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [9, 0, 0, 2, 0, 0, 9, 0, 0, 0], [0, 0, 0, 0, 

# End of document